In [95]:
# Für Nutzung im Notebook
try:
    import evidently
except:
    !pip install git+https://github.com/evidentlyai/evidently.git
        
try:
    import minio
except:
    !pip install minio     
    
try:
    import boto3
except:
    !pip install boto3       

In [96]:
import pandas as pd
import numpy as np
import os
import evidently
import boto3
from sklearn.datasets import fetch_california_housing
from minio import Minio

from evidently import ColumnMapping
from evidently.report import Report
from evidently.metrics.base_metric import generate_column_metrics
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset, DataQualityPreset, RegressionPreset
from evidently.metrics import *
from evidently.test_suite import TestSuite
from evidently.tests.base_test import generate_column_tests
from evidently.test_preset import DataStabilityTestPreset, NoTargetPerformanceTestPreset, RegressionTestPreset
from evidently.tests import *

In [97]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

In [98]:
# df = pd.read_csv("daily_IBM.csv")

In [99]:
# df.rename(columns={'close': 'target'}, inplace=True)
# df['prediction'] = df['target'].values + np.random.normal(0, 5, df.shape[0])

In [100]:
# df[:] = df[::-1]

In [101]:
# df.to_csv("IBM_model1_data1.csv")

In [102]:
data_bucket_url = "logs"
data_file_name = "IBM_model1_data1.csv"
access_key_id = "test"
secret_access_key = "testpassword"
endpoint_url = "http://85.215.53.91:9000"

# Setzen der Umgebungsvariablen für den Zugriff auf Buckets
os.environ["AWS_ACCESS_KEY_ID"] = access_key_id
os.environ["AWS_SECRET_ACCESS_KEY"] = secret_access_key
os.environ["AWS_ENDPOINT_URL"] = endpoint_url

# Laden der Daten aus dem Bucket
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=data_bucket_url, Key=data_file_name)
df = pd.read_csv(obj['Body'])

df



,Unnamed: 0,timestamp,open,high,low,target,volume,prediction
0,0,2023-09-08,147.35,148.5900,147.26,147.68,3722927,154.342693
1,1,2023-09-11,148.57,148.7800,147.58,148.38,3273720,147.786744
2,2,2023-09-12,147.92,148.0000,145.80,146.30,4457695,147.293574
3,3,2023-09-13,145.95,146.9800,145.92,146.55,2627999,150.019108
4,4,2023-09-14,147.38,147.7300,146.48,147.35,2723200,150.924708
...,...,...,...,...,...,...,...,...
95,95,2024-01-25,184.96,196.9000,184.83,190.43,29596239,194.794158
96,96,2024-01-26,191.31,192.3896,186.16,187.42,9895941,189.090067
97,97,2024-01-29,187.46,189.4600,186.05,187.14,6107908,181.246565
98,98,2024-01-30,187.71,188.6500,186.77,187.87,4575058,187.363425


In [103]:
df.drop(df.columns[0], axis=1)

,timestamp,open,high,low,target,volume,prediction
0,2023-09-08,147.35,148.5900,147.26,147.68,3722927,154.342693
1,2023-09-11,148.57,148.7800,147.58,148.38,3273720,147.786744
2,2023-09-12,147.92,148.0000,145.80,146.30,4457695,147.293574
3,2023-09-13,145.95,146.9800,145.92,146.55,2627999,150.019108
4,2023-09-14,147.38,147.7300,146.48,147.35,2723200,150.924708
...,...,...,...,...,...,...,...
95,2024-01-25,184.96,196.9000,184.83,190.43,29596239,194.794158
96,2024-01-26,191.31,192.3896,186.16,187.42,9895941,189.090067
97,2024-01-29,187.46,189.4600,186.05,187.14,6107908,181.246565
98,2024-01-30,187.71,188.6500,186.77,187.87,4575058,187.363425


In [104]:
reference = df.iloc[50:,:]
current = df.iloc[:50,:]

In [109]:
report = Report(metrics=[
    DataDriftPreset(),
    TargetDriftPreset(),
    RegressionPreset()
])

# json report in Bucket speichern
report.run(reference_data=reference, current_data=current)
report_json = report.json()
s3 = boto3.resource('s3')
obj = s3.Object("reports", "/IBM/model1/data1/report.json")
obj.put(Body=report_json)


{'ResponseMetadata': {'RequestId': '17B5074E03E67B6D',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'etag': '"2f55be3404e398b403f3f7b3ab42c8eb"',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '17B5074E03E67B6D',
   'x-content-type-options': 'nosniff',
   'x-xss-protection': '1; mode=block',
   'date': 'Sun, 18 Feb 2024 18:07:45 GMT'},
  'RetryAttempts': 0},
 'ETag': '"2f55be3404e398b403f3f7b3ab42c8eb"'}